# Graph WaveNet on PEMS-BAY (Kaggle)

The official Graph WaveNet code (Wu et al., IJCAI 2019, MIT licence) on PEMS-BAY, as a baseline for HGP. It is run with its **own published recipe**
(Adam, learning rate 0.001, batch 64, dropout 0.3, 100 epochs, forward and backward road graphs plus an adaptive graph; the epoch with the lowest
validation MAE over the 12 output steps is kept). Nothing is tuned.

The reported horizons are 15 / 30 / 60 minutes (output steps 3, 6, 12 of the 12 predicted), and `mean` is the mean of these three, as in HGP.

**Changes to the official code (applied to a scratch copy at run time, nothing else is touched)**
1. `model.py`: three `nn.Conv1d(` become `nn.Conv2d(`. The original fails on current PyTorch (`Conv1d` receives 4-D input); the computation is identical.
2. `train.py`: the commented-out `--seed` option and seeding are switched on, so that several seeds can be run.

**Not the same as the HGP protocol (report this)**: the 70 / 10 / 20 split is over windows (the DCRNN convention), zeros stay as inputs and are masked
in the loss and metrics, one global mean and standard deviation is used, and training is 100 epochs without early stopping.
Metrics come from the official `util.metric` (zero targets masked), so MAPE masks only zero targets.

**Before running**: Accelerator = GPU, Internet = On (the code is cloned), and a PEMS-BAY dataset with `pems-bay.h5` and `adj_mx_bay.pkl` as input.
Use *Save & Run All* so that `/kaggle/working/results_gwnet` is kept.

In [ ]:
# ---- settings
SEEDS = (42,)         # first check seed 42 against the paper; later use (42, 7, 123)
EPOCHS = 100         # the official recipe. Use 40 to match the HGP epoch budget (then say so when reporting)
SHOW_TEST = True     # the recipe is fixed and nothing is selected on test, so the test lines are shown
REPO_URL = "https://github.com/nnzhan/Graph-WaveNet.git"

# official defaults of train.py; listed here only so that training and evaluation use the same values
HP = dict(nhid=32, dropout=0.3, batch_size=64, learning_rate=0.001, weight_decay=0.0001)
NUM_NODES = 325      # PEMS-BAY sensors

## 1. Environment

In [ ]:
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import torch

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(sys.version.split()[0], "| torch", torch.__version__, "|", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if DEVICE == "cpu":
    print("warning: no GPU; one epoch on PEMS-BAY will take very long on CPU")
try:
    import tables  # noqa: F401  (pandas.read_hdf needs it)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tables"], check=True)

## 2. Code and data

In [ ]:
WORK = "/kaggle/working"          # only results_gwnet/ is written here (the saved output)
SCRATCH = "/tmp/gwnet"            # code, generated windows, per-epoch checkpoints and logs
os.makedirs(SCRATCH, exist_ok=True)

CODE = f"{SCRATCH}/gwnet_code"
shutil.rmtree(CODE, ignore_errors=True)
hits = [p for p in glob.glob("/kaggle/input/**/model.py", recursive=True) if "class gwnet" in open(p, errors="ignore").read()]
if hits:
    shutil.copytree(os.path.dirname(hits[0]), CODE)          # the repository was added as an input
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.exists(f"{CODE}/model.py") and os.path.exists(f"{CODE}/train.py"), "Graph WaveNet code not found"

files = [p for p in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(p)]
traffic = [p for p in files if p.lower().endswith(".h5") and "pems" in os.path.basename(p).lower() and "meta" not in os.path.basename(p).lower()]
adjacency = [p for p in files if p.lower().endswith(".pkl") and "adj" in os.path.basename(p).lower()]
assert traffic, "no PEMS-BAY .h5 file under /kaggle/input (expected pems-bay.h5)"
assert adjacency, "no adjacency .pkl under /kaggle/input (expected adj_mx_bay.pkl)"
TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
print("code     :", CODE)
print("traffic  :", TRAFFIC)
print("adjacency:", ADJACENCY)

In [ ]:
def run(cmd, keep=None):
    """Run a command in the code folder; print only the lines for which keep(line) is true (all if None). Returns every line."""
    name = os.path.basename(cmd[1])
    lines = []
    with open(f"{SCRATCH}/log_{name}.txt", "a") as log, subprocess.Popen(
            cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            log.write(line)
            lines.append(line)
            if keep is None or keep(line):
                print(line, end="")
    assert proc.returncode == 0, f"{name} exited with {proc.returncode}; see {SCRATCH}/log_{name}.txt"
    return lines


def patch(path, pairs):
    text = open(path).read()
    for old, new, count in pairs:
        assert text.count(old) == count, f"{path}: expected {count} x {old!r}, found {text.count(old)}"
        text = text.replace(old, new)
    open(path, "w").write(text)


# 1. current PyTorch rejects Conv1d on 4-D input; Conv2d computes the same thing
patch(f"{CODE}/model.py", [("nn.Conv1d(", "nn.Conv2d(", 3)])
# 2. switch on the seed option that the authors commented out
patch(f"{CODE}/train.py", [
    ("#parser.add_argument('--seed',type=int,default=99,help='random seed')", "parser.add_argument('--seed',type=int,default=99,help='random seed')", 1),
    ("#torch.manual_seed(args.seed)", "torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)", 1),
    ("#np.random.seed(args.seed)", "np.random.seed(args.seed)", 1),
])
print("patched model.py (3 x Conv1d -> Conv2d) and train.py (--seed)")

## 3. Generate the PEMS-BAY windows
The repository's own script: 12 steps in, 12 out, speed plus time of day, 70 / 10 / 20 split of the windows.

In [ ]:
DATA_DIR = f"{SCRATCH}/data/PEMS-BAY"
if not all(os.path.exists(f"{DATA_DIR}/{s}.npz") for s in ("train", "val", "test")):
    shutil.rmtree(DATA_DIR, ignore_errors=True)     # the script asks for input() when the folder exists
    run([sys.executable, "generate_training_data.py", "--output_dir", DATA_DIR, "--traffic_df_filename", TRAFFIC])
else:
    print("windows already generated:", DATA_DIR)

# the model needs the true sensor count (the script default is 207 for METR-LA); check it against the generated windows and the graph
import pickle
_n = np.load(f"{DATA_DIR}/test.npz")["x"].shape[2]
_adj = pickle.load(open(ADJACENCY, "rb"), encoding="latin1")[2].shape[0]
assert _n == _adj == NUM_NODES, f"sensor count mismatch: windows {_n}, adjacency {_adj}, NUM_NODES {NUM_NODES}"
print("sensors:", _n)


## 4. Train with the official recipe and evaluate
Each seed writes `results.json`, `eval.json`, `checkpoints/best.pt` and `test_predictions_h{3,6,12}.npz` into `results_gwnet/gwnet__seed<seed>/`.

In [ ]:
EVAL_SCRIPT = r'''
import argparse
import json
import sys

import numpy as np
import torch

ap = argparse.ArgumentParser()
for name, typ in [("code", str), ("ckpt", str), ("data", str), ("adjdata", str), ("num_nodes", int), ("device", str), ("out", str),
                  ("nhid", int), ("dropout", float), ("batch_size", int), ("learning_rate", float), ("weight_decay", float)]:
    ap.add_argument("--" + name, type=typ, required=True)
a = ap.parse_args()
sys.path.insert(0, a.code)
import util
from engine import trainer

device = torch.device(a.device)
_, _, adj = util.load_adj(a.adjdata, "doubletransition")
supports = [torch.tensor(i).to(device) for i in adj]
data = util.load_dataset(a.data, a.batch_size, a.batch_size, a.batch_size)
scaler = data["scaler"]
engine = trainer(scaler, 2, 12, a.num_nodes, a.nhid, a.dropout, a.learning_rate, a.weight_decay, device, supports, True, True, None)
engine.model.load_state_dict(torch.load(a.ckpt, map_location=device))
engine.model.eval()
HORIZONS = {3: 2, 6: 5, 12: 11}    # 15 / 30 / 60 minutes -> index of the 12 output steps


def evaluate(split):
    realy = torch.Tensor(data["y_" + split]).to(device).transpose(1, 3)[:, 0, :, :]
    outputs = []
    for x, _ in data[split + "_loader"].get_iterator():
        with torch.no_grad():
            outputs.append(engine.model(torch.Tensor(x).to(device).transpose(1, 3)).transpose(1, 3).squeeze(1))
    yhat = torch.cat(outputs, 0)[: realy.size(0)]
    result, arrays = {}, {}
    for h, i in HORIZONS.items():
        pred, real = scaler.inverse_transform(yhat[:, :, i]), realy[:, :, i]
        mae, mape, rmse = util.metric(pred, real)
        result[str(h)] = {"mae": mae, "rmse": rmse, "mape": 100 * mape}
        arrays[h] = (real.cpu().numpy().astype(np.float32), pred.cpu().numpy().astype(np.float32))
    result["mean"] = {k: float(np.mean([result[str(h)][k] for h in HORIZONS])) for k in ("mae", "rmse", "mape")}
    return result, arrays


validation, _ = evaluate("val")
test, arrays = evaluate("test")
for h, (targets, predictions) in arrays.items():
    np.savez_compressed(f"{a.out}/test_predictions_h{h}.npz", targets=targets, predictions=predictions)
json.dump({"parameters": sum(p.numel() for p in engine.model.parameters()), "validation": validation, "test": test},
          open(f"{a.out}/eval.json", "w"), indent=2)
print("evaluated", a.ckpt)
'''
open(f"{SCRATCH}/gwnet_eval.py", "w").write(EVAL_SCRIPT)

In [ ]:
def shown(line):
    s = line.strip()
    if s.startswith("Epoch:") and "Valid Loss" in s:
        return int(s.split()[1].rstrip(",")) % 10 == 0
    if s.startswith(("Average Training Time", "Training finished", "Total time")):
        return True
    return SHOW_TEST and (s.startswith("On average") or any(s.startswith(f"Evaluate best model on test data for horizon {h}:") or
                          s.startswith(f"Evaluate best model on test data for horizon {h},") for h in (3, 6, 12)))


EPOCH = re.compile(r"Epoch: (\d+), Train Loss: \S+, Train MAPE: \S+, Train RMSE: \S+, Valid Loss: (\S+), Valid MAPE: \S+, Valid RMSE: \S+, Training Time: (\S+)/epoch")

for seed in SEEDS:
    out = f"{WORK}/results_gwnet/gwnet__seed{seed}"
    if os.path.exists(f"{out}/results.json"):
        print("already done: seed", seed)
        continue
    print("--- seed", seed)
    ckdir = f"{SCRATCH}/ckpt/seed{seed}"
    shutil.rmtree(ckdir, ignore_errors=True)
    os.makedirs(ckdir)
    os.makedirs(f"{out}/checkpoints", exist_ok=True)
    t0 = time.time()
    lines = run([sys.executable, "train.py", "--device", DEVICE, "--data", DATA_DIR, "--adjdata", ADJACENCY, "--num_nodes", str(NUM_NODES),
                 "--gcn_bool", "--adjtype", "doubletransition", "--addaptadj", "--randomadj", "--seed", str(seed), "--epochs", str(EPOCHS),
                 "--nhid", str(HP["nhid"]), "--dropout", str(HP["dropout"]), "--batch_size", str(HP["batch_size"]),
                 "--learning_rate", str(HP["learning_rate"]), "--weight_decay", str(HP["weight_decay"]),
                 "--save", f"{ckdir}/m", "--expid", str(seed)], keep=shown)
    history = [(int(m.group(1)), float(m.group(2)), float(m.group(3))) for m in map(EPOCH.search, lines) if m]
    assert history, "no epoch lines found in the training log"
    best_epoch = min(history, key=lambda r: r[1])[0]
    ckpt = glob.glob(f"{ckdir}/m_epoch_{best_epoch}_*.pth")
    assert ckpt, f"checkpoint of epoch {best_epoch} not found"
    shutil.copy(ckpt[0], f"{out}/checkpoints/best.pt")
    run([sys.executable, f"{SCRATCH}/gwnet_eval.py", "--code", CODE, "--ckpt", ckpt[0], "--data", DATA_DIR, "--adjdata", ADJACENCY,
         "--num_nodes", str(NUM_NODES), "--device", DEVICE, "--out", out, *[a for k, v in HP.items() for a in (f"--{k}", str(v))]],
        keep=lambda l: l.startswith("evaluated"))
    ev = json.load(open(f"{out}/eval.json"))
    results = {"run": f"gwnet__seed{seed}", "parameters": ev["parameters"], "best_epoch": best_epoch, "epochs": len(history),
               "validation": ev["validation"], "test": ev["test"], "training_seconds": sum(r[2] for r in history),
               "wall_seconds": time.time() - t0, "device": DEVICE, "recipe": {"epochs": EPOCHS, **HP, "seed": seed}}
    json.dump(results, open(f"{out}/results.json", "w"), indent=2)
    v = ev["validation"]
    print(f"done: best epoch {best_epoch} of {len(history)}, val MAE 15/30/60 min {v['3']['mae']:.3f} {v['6']['mae']:.3f} {v['12']['mae']:.3f} "
          f"(mean {v['mean']['mae']:.3f}), {results['training_seconds'] / 60:.1f} min training, {ev['parameters']:,} parameters")

## 5. Summary
Validation and test metrics at 15 / 30 / 60 minutes for every finished seed; mean and standard deviation over seeds.

In [ ]:
runs = [json.load(open(f)) for f in sorted(glob.glob(f"{WORK}/results_gwnet/*/results.json"))]
H = ("3", "6", "12")


def line(label, values):
    a = np.array(values)
    return f"{label:28s} " + " ".join(f"{(f'{a[:, i].mean():.3f}' + (f' ± {a[:, i].std(ddof=1):.3f}' if len(a) > 1 else '')):>16s}" for i in range(a.shape[1]))


print(f"{'':28s} " + " ".join(f"{h:>16s}" for h in ("15 min", "30 min", "60 min", "mean")))
for split in ("validation", "test"):
    for metric in ("mae", "rmse", "mape"):
        print(line(f"{split} {metric.upper()}", [[r[split][h][metric] for h in H] + [r[split]["mean"][metric]] for r in runs]))
print()
for r in runs:
    print(f"{r['run']:20s} best epoch {r['best_epoch']:3d} of {r['epochs']:3d}   {r['training_seconds'] / 60:6.1f} min training   {r['parameters']:,} parameters   {r['device']}")